# Stock Trading Agent — Evaluation with Langfuse

This notebook builds a **stock trading assistant** with three tools and evaluates it
using Langfuse datasets and custom scoring.

| Section | What you will learn |
|---------|--------------------|  
| **Tools** | `get_price`, `get_news` (Tavily), `buy_ticker` (mock) |
| **Agent** | ReAct agent with `MemorySaver` checkpointer for multi-turn conversations |
| **Evaluation** | Create a Langfuse dataset, define custom evaluators, and run experiment |

## Setup

1. Run the setup cell below first — all other cells depend on it.
2. Set `MODEL_PROVIDER = "openai"` or `"gemini"`.
3. Required keys in `.env`: `LANGFUSE_SECRET_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_HOST` (or `LANGFUSE_BASE_URL`), `TAVILY_API_KEY`, plus the provider key.
4. If packages are missing:
   ```bash
   uv add langfuse yfinance tavily-python langgraph
   ```

In [1]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langfuse import Langfuse
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert LANGFUSE_SECRET_KEY, "LANGFUSE_SECRET_KEY not found — add it to your .env file"
assert LANGFUSE_PUBLIC_KEY, "LANGFUSE_PUBLIC_KEY not found — add it to your .env file"
assert TAVILY_API_KEY, "TAVILY_API_KEY not found — add it to your .env file"

# ── Set env vars for Langfuse SDK auto-detection ─────────────────────────────
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST

# ── Initialize Langfuse client ────────────────────────────────────────────────
langfuse = Langfuse(
    secret_key=LANGFUSE_SECRET_KEY,
    public_key=LANGFUSE_PUBLIC_KEY,
    host=LANGFUSE_HOST,
)

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

INFO | Setup complete — LLM provider: openai


---
## Tools

| Tool | Description |
|------|-------------|
| `get_price` | Fetches historical stock prices via Yahoo Finance for a given ticker and date range |
| `get_news` | Searches for related news using Tavily (returns top 3 results) |
| `buy_ticker` | Mock buy-order tool — returns a confirmation string |

In [2]:
@tool
def get_price(ticker: str, date_range: str) -> str:
    """Get stock price for a ticker over a date range.

    Args:
        ticker: Stock ticker symbol (e.g. AAPL, GOOGL, TSLA).
        date_range: Yahoo Finance period string — e.g. '1d', '5d', '1mo', '3mo', '1y'.
    Returns:
        Price summary including latest close, high, low, and period change.
    """
    logger.info("get_price called | ticker=%s  date_range=%s", ticker, date_range)
    stock = yf.Ticker(ticker)
    hist = stock.history(period=date_range)
    if hist.empty:
        return f"Could not fetch price data for {ticker} over {date_range}."

    latest_close = hist["Close"].iloc[-1]
    period_high = hist["High"].max()
    period_low = hist["Low"].min()
    first_close = hist["Close"].iloc[0]
    change = latest_close - first_close
    pct = (change / first_close) * 100

    result = (
        f"{ticker} ({date_range}): latest close ${latest_close:.2f}, "
        f"high ${period_high:.2f}, low ${period_low:.2f}, "
        f"change {change:+.2f} ({pct:+.2f}%)"
    )
    logger.info("get_price result: %s", result)
    return result


logger.info("Tool ready: get_price")

INFO | Tool ready: get_price


In [3]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def get_news(ticker: str, search_term: str) -> str:
    """Search for recent news related to a stock ticker.

    Args:
        ticker: Stock ticker symbol (e.g. AAPL, NVDA).
        search_term: Additional keywords to narrow the search.
    Returns:
        Top 3 news results with title, snippet, and URL.
    """
    query = f"{ticker} {search_term}"
    logger.info("get_news called | query='%s'", query)

    response = tavily_client.search(query=query, max_results=3)
    results = response.get("results", [])

    if not results:
        return f"No news found for '{query}'."

    lines = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "No title")
        snippet = r.get("content", "")[:200]
        url = r.get("url", "")
        lines.append(f"{i}. {title}\n   {snippet}\n   {url}")

    result = "\n\n".join(lines)
    logger.info("get_news returned %d results", len(results))
    return result


logger.info("Tool ready: get_news")

INFO | Tool ready: get_news


In [4]:
@tool
def buy_ticker(ticker: str, amount: float, target_price: float) -> str:
    """Place buy order for a stock.

    Args:
        ticker: Stock ticker symbol to buy.
        amount: Number of shares to buy.
        target_price: Target price per share for the limit order.
    Returns:
        Confirmation message for the buy order.
    """
    logger.info(
        "buy_ticker called | ticker=%s  amount=%.2f  target_price=%.2f",
        ticker, amount, target_price,
    )
    result = (
        f"Done, a buy order has been set for {ticker} "
        f"with amount of {amount} shares at price ${target_price:.2f}"
    )
    logger.info("buy_ticker result: %s", result)
    return result


logger.info("Tool ready: buy_ticker")

INFO | Tool ready: buy_ticker


---
## Build the Agent

We use `create_react_agent` from LangGraph with a `MemorySaver` checkpointer
so the agent can maintain conversation context across turns.

Each conversation is isolated by `thread_id` in the config.

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = (
    "You are a stock trading assistant. You can look up stock prices, "
    "search for related news, and place mock buy orders. "
    "Always use the tools when the user asks about prices, news, or wants to buy. "
    "Be concise and include specific numbers in your answers."
)

tools = [get_price, get_news, buy_ticker]
checkpointer = MemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

logger.info("Agent created with tools: %s", [t.name for t in tools])

INFO | Agent created with tools: ['get_price', 'get_news', 'buy_ticker']


---
## Test the Agent

Run a few queries to verify each tool works correctly.
We use the same `thread_id` across turns so the agent remembers context.

In [6]:
config = {"configurable": {"thread_id": "test-session-1"}}

# ── Test 1: Price lookup ───────────────────────────────────────────────────────
result = agent.invoke(
    {"messages": [HumanMessage(content="What is the price of NVDA over the last 5 days?")]},
    config=config,
)
print("=== Price Lookup ===")
print(result["messages"][-1].content)

INFO | get_price called | ticker=NVDA  date_range=5d
INFO | get_price result: NVDA (5d): latest close $175.75, high $177.37, low $164.27, change +4.51 (+2.63%)


=== Price Lookup ===
Over the last 5 days, NVIDIA (NVDA) has had the following price summary:
- Latest Close: $175.75
- High: $177.37
- Low: $164.27
- Change: +$4.51 (+2.63%)


In [7]:
# ── Test 2: News search ────────────────────────────────────────────────────────
result2 = agent.invoke(
    {"messages": [HumanMessage(content="Find me news about TSLA related to earnings")]},
    config=config,
)
print("=== News Search ===")
print(result2["messages"][-1].content)

INFO | get_news called | query='TSLA earnings'
INFO | get_news returned 3 results


=== News Search ===
Here are some recent news articles about Tesla (TSLA) related to earnings:

1. **Tesla (TSLA) Earnings: Latest Report, Earnings Call & Financials**  
   [Read more](https://public.com/stocks/tsla/earnings)

2. **Tesla, Inc. (TSLA) Earnings Report Date - Nasdaq**  
   [Read more](https://www.nasdaq.com/market-activity/stocks/tsla/earnings)

3. **Tesla Earnings Calls - YouTube**  
   [Watch here](https://www.youtube.com/playlist?list=PLEox0nUMFPF4TOgwz6PYqoV_lhyENbWr6) 

Feel free to check these links for detailed information!


In [8]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Should I buy it")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

=== Buy Order ===
To provide a recommendation, I can check the current price of TSLA and any recent news that might influence your decision. Would you like me to do that?


In [9]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Yes")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | get_price called | ticker=TSLA  date_range=1d
INFO | get_news called | query='TSLA latest'
INFO | get_price result: TSLA (1d): latest close $381.26, high $383.14, low $374.08, change +0.00 (+0.00%)
INFO | get_news returned 3 results


=== Buy Order ===
As of the latest data, here are the details for Tesla (TSLA):

- **Latest Close**: $381.26
- **High**: $383.14
- **Low**: $374.08
- **Change**: $0.00 (0.00%)

### Recent News:
1. **TSLA Stock Quote Price and Forecast - CNN**  
   [Read more](https://www.cnn.com/markets/stocks/TSLA)

2. **Stock Price & Latest News - Reuters**  
   [Read more](https://www.reuters.com/markets/companies/TSLA.O/)

3. **Tesla: TSLA Stock Price Quote & News | Robinhood**  
   Current price is $380.90, with a P/E ratio of 345.69. [Read more](https://robinhood.com/us/en/stocks/TSLA/)

### Recommendation:
- The stock is currently stable with no significant change. However, the high P/E ratio suggests that it may be overvalued compared to its earnings.
- Consider your investment strategy, risk tolerance, and whether you believe in Tesla's long-term growth potential before making a decision.

Would you like to proceed with a mock buy order, or do you need more information?


In [10]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Buy 100 shares of it at $370")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | buy_ticker called | ticker=TSLA  amount=100.00  target_price=370.00
INFO | buy_ticker result: Done, a buy order has been set for TSLA with amount of 100.0 shares at price $370.00


=== Buy Order ===
Your buy order for 100 shares of Tesla (TSLA) at $370.00 has been successfully set. If you need any further assistance, feel free to ask!


---
## Evaluation with Langfuse

We will:
1. **Create a dataset** with example queries and expected outputs
2. **Define evaluators** — tool-use check + LLM-as-judge for correctness (returning `Evaluation` objects)
3. **Run `dataset.run_experiment()`** and view results in Langfuse

### 1. Create (or load) the Dataset

In [11]:
DATASET_NAME = "stock-trading-agent-eval"

examples = [
    {
        "input": {"question": "What is the price of AAPL over the last month?"},
        "expected_output": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain AAPL price data including close price, high, low, and percentage change over the last month.",
        },
    },
    {
        "input": {"question": "Find news about NVDA related to AI chips"},
        "expected_output": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about NVDA related to AI chips, including titles and links.",
        },
    },
    {
        "input": {"question": "Buy 50 shares of TSLA at $250"},
        "expected_output": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for TSLA with 50 shares at $250.",
        },
    },
    {
        "input": {"question": "What is GOOGL stock price for the past 5 days?"},
        "expected_output": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain GOOGL price data over 5 days with close, high, low, and change.",
        },
    },
    {
        "input": {"question": "Search for recent AAPL news about iPhone sales"},
        "expected_output": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about AAPL and iPhone sales.",
        },
    },
    {
        "input": {"question": "Place a buy order for 100 shares of AAPL at $190"},
        "expected_output": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for AAPL with 100 shares at $190.",
        },
    },
]

# Create dataset (idempotent — returns existing dataset if same name)
dataset = langfuse.create_dataset(
    name=DATASET_NAME,
    description="Evaluation dataset for stock trading agent with price, news, and buy queries.",
)

for ex in examples:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=ex["input"],
        expected_output=ex["expected_output"],
    )

logger.info("Dataset '%s' created with %d examples", DATASET_NAME, len(examples))

INFO | Dataset 'stock-trading-agent-eval' created with 6 examples


### 2. Define Evaluators

We define two evaluators that return `Evaluation` objects:

| Evaluator | What it checks |
|-----------|---------------|
| `correct_tool_used` | Did the agent call the expected tool? (from intermediate messages) |
| `answer_correctness` | LLM-as-judge — does the final answer match the expected criteria? |

In [12]:
from langfuse import Evaluation


def correct_tool_used(*, output, expected_output, **kwargs):
    """Check whether the agent called the expected tool."""
    expected_tool = expected_output.get("expected_tool", "")
    messages = output.get("messages", [])

    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                if tc["name"] == expected_tool:
                    logger.debug("Found expected tool call: %s", expected_tool)
                    return Evaluation(
                        name="correct_tool_used",
                        value=1.0,
                        comment=f"Called {expected_tool}",
                    )

    logger.debug("Expected tool '%s' was NOT called", expected_tool)
    return Evaluation(
        name="correct_tool_used",
        value=0.0,
        comment=f"Did not call {expected_tool}",
    )


judge_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)


def answer_correctness(*, output, expected_output, **kwargs):
    """LLM-as-judge: does the final answer satisfy the expected criteria?"""
    actual_answer = output["messages"][-1].content
    expected = expected_output.get("expected_answer", "")

    instructions = (
        "You are an evaluation judge. Given an actual answer from a stock trading agent "
        "and the expected criteria, determine whether the actual answer satisfies the criteria. "
        "Respond ONLY with 'CORRECT' or 'INCORRECT'."
    )
    user_msg = f"ACTUAL ANSWER:\n{actual_answer}\n\nEXPECTED CRITERIA:\n{expected}"

    response = judge_llm.invoke(
        [
            {"role": "system", "content": instructions},
            {"role": "user", "content": user_msg},
        ]
    )
    is_correct = response.content.strip().upper() == "CORRECT"
    return Evaluation(
        name="answer_correctness",
        value=1.0 if is_correct else 0.0,
        comment=f"Judge: {'CORRECT' if is_correct else 'INCORRECT'}",
    )


logger.info("Evaluators defined: correct_tool_used, answer_correctness")

INFO | Evaluators defined: correct_tool_used, answer_correctness


### 3. Run Evaluation

We create a stateless agent (no checkpointer) so each evaluation example runs independently.
Then we call `dataset.run_experiment()` which handles:
- Concurrent execution with configurable limits
- Automatic tracing of all executions
- Running evaluators on each item's output
- Linking traces to dataset items for comparison in the Langfuse UI

In [13]:
eval_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)


def agent_task(*, item, **kwargs):
    """Adapter: convert dataset item to agent invocation."""
    return eval_agent.invoke(
        {"messages": [{"role": "user", "content": item.input["question"]}]},
    )


dataset = langfuse.get_dataset(DATASET_NAME)

experiment_results = dataset.run_experiment(
    name="stock-agent-eval",
    task=agent_task,
    evaluators=[correct_tool_used, answer_correctness],
    max_concurrency=2,
)

logger.info("Evaluation complete — check results in Langfuse dashboard")

INFO | buy_ticker called | ticker=AAPL  amount=100.00  target_price=190.00
INFO | buy_ticker result: Done, a buy order has been set for AAPL with amount of 100.0 shares at price $190.00
INFO | get_news called | query='AAPL iPhone sales'
INFO | get_news returned 3 results
INFO | get_price called | ticker=GOOGL  date_range=5d
INFO | get_price result: GOOGL (5d): latest close $297.39, high $300.52, low $272.11, change +16.47 (+5.86%)
INFO | buy_ticker called | ticker=TSLA  amount=50.00  target_price=250.00
INFO | buy_ticker result: Done, a buy order has been set for TSLA with amount of 50.0 shares at price $250.00
INFO | get_news called | query='NVDA AI chips'
INFO | get_news returned 3 results
INFO | get_price called | ticker=AAPL  date_range=1mo
INFO | get_price result: AAPL (1mo): latest close $255.63, high $266.53, low $245.51, change -9.09 (-3.43%)
INFO | Evaluation complete — check results in Langfuse dashboard


In [14]:
print(experiment_results.format())

Individual Results: Hidden (6 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: stock-agent-eval
📋 Run name: stock-agent-eval - 2026-04-02T06:44:15.243297Z
6 items
Evaluations:
  • answer_correctness
  • correct_tool_used

Average Scores:
  • answer_correctness: 1.000
  • correct_tool_used: 1.000

🔗 Dataset Run:
   https://cloud.langfuse.com/project/cmnag0e650018ad08ois3cty5/datasets/cmnh401a500r7ad069p2j3pqi/runs/a8b9dd79-a80d-460a-9269-122c066f451d
